# 18_descriptors_for_weka — 1:1 학습셋 descriptor 계산 & WEKA용 CSV 준비

1:1 학습셋(active + 실측 inactive + decoy)에 RDKit 2D descriptor 217종을 계산해
(1) 전체 Excel과 (2) WEKA feature selection용 CSV를 만든다.

**WEKA CSV 규칙:**
- 첫 줄 = 변수명, 각 열 = 수치형 descriptor
- 클래스 변수 `potency`(1=active/0=inactive)는 **마지막 열**
- 문자열 컬럼(SMILES·InChIKey·source) 삭제, **결측값 없이** 정제

In [ ]:
# 프로젝트 루트로 이동 + import
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')
print('작업 폴더:', os.getcwd())

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

### 1:1 학습셋 로드

In [ ]:
# 1:1 학습셋 로드 (active + 실측 inactive + decoy). label(1/0) -> potency
SRC = 'data/train_1to1.csv'
df = pd.read_csv(SRC)
df = df.rename(columns={'label': 'potency'})
print('[0] 1:1 학습셋 shape:', df.shape)
print('    potency 분포:', dict(df.potency.value_counts()))
print('    source 분포:', dict(df.source.value_counts()))

### RDKit descriptor 217종 계산 → 전체 Excel 저장

In [ ]:
# RDKit 2D descriptor 217종 계산 -> 전체 Excel 저장(메타 + potency + descriptor)
desc_names = [n for n, _ in Descriptors._descList]
print('descriptor', len(desc_names), '종 계산 중... (수 분 소요)')

rows, keep = [], []
for i, smi in enumerate(df['canonical_smiles']):
    m = Chem.MolFromSmiles(str(smi))
    if m is None:
        continue
    d = Descriptors.CalcMolDescriptors(m)
    rows.append([d.get(n, np.nan) for n in desc_names])
    keep.append(i)
    if (i + 1) % 1000 == 0:
        print('  ', i + 1, '/', len(df))

X = pd.DataFrame(rows, columns=desc_names)
meta = df.iloc[keep][['canonical_smiles', 'inchikey', 'source', 'potency']].reset_index(drop=True)
full = pd.concat([meta, X.reset_index(drop=True)], axis=1)

XLSX = 'data/HSD17B13_1to1_descriptors.xlsx'
full.to_excel(XLSX, index=False)
print('[Excel] 전체 저장:', XLSX, '| shape', full.shape,
      '(메타 4 + descriptor', len(desc_names), ')')

### WEKA용 CSV 정제
숫자형 descriptor만 남기고, inf/결측이 있는 행을 제거해 완전한 수치 행렬로 만든 뒤,
클래스 `potency`를 **마지막 열**에 붙여 CSV로 저장한다. (VarianceThreshold 등 실제 선택은 WEKA에서)

In [ ]:
# ===== WEKA용 CSV 정제 =====
# 규칙: 숫자형 descriptor만 + 클래스(potency)는 '마지막 열' + 문자열 컬럼 삭제 + 결측 없음
compound_info = full[['canonical_smiles', 'inchikey', 'source', 'potency']]  # 메타(문자열 등)
descriptor_data = full[desc_names]                                            # descriptor만
print('[1] descriptor 데이터 shape:', descriptor_data.shape)

# 1) 숫자로 변환 안 되는(문자/이상값) 컬럼 제거
def remove_invalid_descriptors(data):
    invalid = []
    for col in data.columns:
        try:
            pd.to_numeric(data[col], errors='raise')
        except Exception:
            invalid.append(col)
    print('[2] 숫자변환 실패(문자/이상) 컬럼 제거:', len(invalid), '개')
    return data.drop(columns=invalid)

desc_num = remove_invalid_descriptors(descriptor_data).apply(pd.to_numeric)

# 2) inf -> NaN 후 결측 처리: NaN 있는 '행' 제거(모든 descriptor 컬럼 유지)
desc_num = desc_num.replace([np.inf, -np.inf], np.nan)
n_nan_cell = int(desc_num.isna().sum().sum())
nan_row_mask = desc_num.isna().any(axis=1)
print('[3] 결측/inf 셀', n_nan_cell, '개 -> 결측 포함 행', int(nan_row_mask.sum()), '개 제거')
desc_num = desc_num[~nan_row_mask].reset_index(drop=True)
potency = compound_info['potency'][~nan_row_mask.values].reset_index(drop=True)

# 3) 클래스(potency)를 '마지막 열'에 두고, 문자열 컬럼은 모두 제외
weka_df = desc_num.copy()
weka_df['potency'] = potency.values         # 마지막 열
print('[4] WEKA용 shape (descriptor + potency):', weka_df.shape)
print('    마지막 열:', weka_df.columns[-1], '| 클래스 분포:', dict(weka_df.potency.value_counts()))

CSV = 'data/HSD17B13_1to1_descriptors_weka.csv'
weka_df.to_csv(CSV, index=False)
print('[5] WEKA용 CSV 저장 완료:', CSV)
print('    (WEKA Explorer -> Open file -> Select attributes 탭에서 사용)')